
# Multiple Linear Regression

## Beyond the Simple Line

In the real world, a single variable is rarely enough to predict an outcome. House prices depend on size, location, **and** condition. Health depends on diet, exercise, **and** genetics.

**Multiple Linear Regression** extends the simple model to accommodate multiple predictors ($x_1, x_2, \dots, x_n$). Instead of fitting a **Line** in 2D space, we fit a **Hyperplane** in multidimensional space.

## The Mathematical Model

The equation expands to sum the contributions of each feature:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n + \epsilon$$

Where:

-   $\beta_0$: The Intercept (The baseline value when all features are 0).
-   $\beta_i$: The Coefficient (The change in $y$ for a 1-unit increase in $x_i$, **holding all other variables constant**).

## Assumptions (Extended)

Multiple Regression relies on the **same core assumptions** as Simple Linear Regression, and adds an additional one:

1.  **Linearity**: The relationship between each predictor and $y$ is linear.
2.  **Independence**: Observations are independent of each other.
3.  **Homoscedasticity**: Constant variance of errors across all predicted values.
4.  **Normality**: Errors are normally distributed (important for inference).
5.  **No Multicollinearity**: The predictors ($x_i$) should not be highly correlated with **each other**.

**Multicollinearity**: In simple regression with one predictor, multicollinearity cannot exist by definition. With multiple predictors, it becomes a real concern. If $x_1$ and $x_2$ are highly correlated, the model cannot reliably determine which one is truly affecting $y$. This leads to:

-   Unstable, unreliable coefficients
-   Large standard errors
-   Coefficients that flip signs or change dramatically with small data changes

## Practical Demonstration

We will generate a synthetic dataset with 2 features so we can visualize the model as a **3D Plane**.

### Generate 3D Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
#  from mpl_toolkits.mplot3d import Axes3D

# Generate data: 100 samples, 2 features, 1 target
X, y, coef = make_regression(n_samples=100, n_features=2, noise=10, coef=True, random_state=42)

# Wrap in DataFrame
df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])
df['Target'] = y

print(f"True Coefficients: {coef}")
print(df.head())

### Correlation Analysis

Before modeling, check if features are correlated with the target (Good) or with each other (Bad - Multicollinearity).

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()

### Detecting Multicollinearity with VIF

The correlation matrix gives a quick visual check, but **Variance Inflation Factor (VIF)** is the standard diagnostic for multicollinearity.

VIF measures how much the variance of a coefficient is "inflated" due to correlation with other features:

$$\text{VIF}_i = \frac{1}{1 - R^2_i}$$

Where $R^2_i$ is the R² from regressing feature $x_i$ against all other features.

**Rule of Thumb**:

-   VIF = 1: No correlation (ideal)
-   VIF < 5: Acceptable
-   VIF > 5-10: Problematic multicollinearity
-   VIF > 10: Severe — consider removing or combining features

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df, features):
    """Calculate VIF for each feature in the dataframe."""
    vif_data = pd.DataFrame()
    vif_data['Feature'] = features
    vif_data['VIF'] = [
        variance_inflation_factor(df[features].values, i)
        for i in range(len(features))
    ]
    return vif_data

# Calculate VIF for our features
features = ['Feature_1', 'Feature_2']
vif_results = calculate_vif(df, features)
print("Variance Inflation Factors:")
print(vif_results.to_string(index=False))

**Interpretation**: Since our synthetic data was generated with independent features, VIF values should be close to 1. In real-world datasets, high VIF indicates you may need to:

1.  Remove one of the correlated features
2.  Combine correlated features (e.g., via PCA)
3.  Use regularization (Ridge/Lasso regression)

### Train and Visualize (3D)

We fit the model and plot the resulting plane.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Split
X_train, X_test, y_train, y_test = train_test_split(df[['Feature_1', 'Feature_2']], df['Target'], test_size=0.2, random_state=42)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept: {model.intercept_:.2f}")
print(f"Coefficients: {model.coef_}")

# --- 3D Visualization ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot Data Points
ax.scatter(X_test['Feature_1'], X_test['Feature_2'], y_test, color='red', label='Actual Data')

# Plot the Plane
x0_surf = np.linspace(X_test['Feature_1'].min(), X_test['Feature_1'].max(), 20)
x1_surf = np.linspace(X_test['Feature_2'].min(), X_test['Feature_2'].max(), 20)
X0, X1 = np.meshgrid(x0_surf, x1_surf)
Y_pred_surf = model.intercept_ + model.coef_[0] * X0 + model.coef_[1] * X1

ax.plot_surface(X0, X1, Y_pred_surf, color='blue', alpha=0.3)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Target')
ax.set_title('Multiple Linear Regression Plane')
plt.show()

### Evaluation and Residual Analysis

We check the standard metrics.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.2f}")
print(f"R²:  {r2:.4f}")

# Residual Plot (Homoscedasticity Check)
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot (Check for random scatter)")
plt.show()

### Interpreting Coefficients: The Scale Problem

The coefficients we printed earlier have a hidden issue: **they are not directly comparable** when features have different scales.

For example, if Feature<sub>1</sub> is measured in meters and Feature<sub>2</sub> in kilometers, a coefficient of 50 for Feature<sub>1</sub> might actually represent a smaller effect than a coefficient of 2 for Feature<sub>2</sub>.

**Standardized Coefficients** solve this by expressing coefficients in terms of standard deviations:

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize features (mean=0, std=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train on standardized data
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)

# Compare coefficients
print("Raw Coefficients (original scale):")
print(f"  Feature_1: {model.coef_[0]:.4f}")
print(f"  Feature_2: {model.coef_[1]:.4f}")
print()
print("Standardized Coefficients (comparable):")
print(f"  Feature_1: {model_scaled.coef_[0]:.4f}")
print(f"  Feature_2: {model_scaled.coef_[1]:.4f}")

**Interpretation**:

-   **Raw coefficients**: "A 1-unit increase in $x_i$ leads to a $\beta_i$ change in $y$"
-   **Standardized coefficients**: "A 1 standard deviation increase in $x_i$ leads to a $\beta_i$ standard deviation change in $y$"

Standardized coefficients let you answer: **"Which feature has the strongest influence on the prediction?"** — something raw coefficients cannot reliably tell you.

## Exercises

You will now deal with a higher-dimensional dataset.

### Generate 3-Feature Data

Generate a dataset with **3 features** and slightly higher noise.

-   `n_samples` = 200
-   `n_features` = 3
-   `noise` = 20

### Feature Selection & Training

Train a model using **only** Feature 1 and Feature 3. Ignore Feature 2. Evaluate the $R^2$.

### The "Kitchen Sink" Model

Now train a model using **all 3 features**. Compare the $R^2$. Does it increase?

### The Problem with R²

**Observation**: The $R^2$ is higher (or the same). But this reveals a trap!

**Training** $R^2$ will **always** increase (or stay the same) when you add features - even useless ones. The model simply has more knobs to turn to fit the data. This doesn't mean the model **generalizes** better. On **test** data, adding irrelevant features can actually hurt performance due to overfitting.

### Adjusted R²: The Honest Metric

**Adjusted R²** penalizes the addition of features that don't improve the model proportionally:

$$R^2_{adj} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

Where:

-   $n$ = number of samples
-   $p$ = number of features

If a new feature doesn't contribute enough explanatory power, Adjusted $R^2$ will **decrease**.

**Key Insight**: If the Adjusted $R^2$ doesn't improve (or decreases) when adding a feature, that feature likely isn't contributing meaningful information. This is a simple form of **feature selection**.

### Non-Linearity Challenge

Modify the target variable to be non-linear: $y = 3x_1^2 + 2x_2 + \text{noise}$. Fit a linear model and examine the Residual Plot to see why linear models fail on non-linear data. **Observation**: You should see a distinct "U-shape" (parabola) in the residuals. This systematic pattern proves the linear model is insufficient—it's missing the quadratic relationship in the data.

## Summary

1.  **Hyperplanes**: Multiple Regression fits a plane, not a line.
2.  **Multicollinearity**: Watch out for correlated features; they destabilize the model.
3.  **Coefficients**: In multiple regression, a coefficient represents the effect of a feature **assuming all other features stay constant**.
4.  **Adjusted R²**: Penalizes unnecessary features; use it to compare models.
5.  **Standardized Coefficients**: Required to compare feature importance across different scales.
6.  **Beyond Prediction**: For formal hypothesis testing about coefficients (`p`-values, confidence intervals), see `statsmodels.OLS`. This becomes important when the goal is inference rather than prediction.